In [ ]:
import csv
import time
from pathlib import Path
from typing import Dict, List

import requests
from bs4 import BeautifulSoup

# STEP 2 ONLY: from existing article list CSV -> detailed article-level data
# Input CSV (existing): Journal_of_IT.csv
#   columns: Title (issue label), Issue_Title (article title), Issue_URL (URL)
# Output CSV: JIT_article_data.csv
#   columns: URL,Journal_Title,Article_Title,Volume_Issue,Month_Year,
#            Abstract,Keywords,Author_name,Author_email,Author_Address

BASE_URL = "https://journals.sagepub.com"
JOURNAL_NAME = "Journal of Information Technology"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


def get_soup(url: str) -> BeautifulSoup:
    """Fetch URL and return BeautifulSoup."""
    for attempt in range(3):
        resp = requests.get(url, headers=HEADERS, timeout=20)
        if resp.status_code == 200:
            return BeautifulSoup(resp.text, "html.parser")
        time.sleep(2 + attempt)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def parse_article_page(url: str) -> Dict[str, str]:
    """Parse one article page and return abstract/keywords/author data."""
    soup = get_soup(url)

    # ABSTRACT
    abstract_text = ""
    abstract_selectors = [
        "section.abstractSection > p",
        "div.abstractSection > p",
        "section#abstract > p",
        "div#abstract > p",
        "div.article-section__content > p",
    ]
    for sel in abstract_selectors:
        paras = soup.select(sel)
        if paras:
            abstract_text = " ".join(p.get_text(" ", strip=True) for p in paras)
            break

    # KEYWORDS
    keywords_list: List[str] = []
    kw_containers = soup.select("div.keyword, div.hlFld-KeywordText, p.keywords")
    if kw_containers:
        text = " ".join(k.get_text(" ", strip=True) for k in kw_containers)
        # try splitting on ; or ,
        for sep in [";", ","]:
            if sep in text:
                parts = [x.strip() for x in text.split(sep)]
                keywords_list = [p for p in parts if p]
                break
        if not keywords_list and text.strip():
            keywords_list = [text.strip()]
    keywords = "; ".join(dict.fromkeys(keywords_list))  # unique, keep order

    # AUTHORS: names, emails, addresses
    author_names: List[str] = []
    author_emails: List[str] = []
    author_addresses: List[str] = []

    # Names
    name_selectors = [
        "a.entryAuthor",
        "span.contributor-name",
        "span.author-name",
        "li.author",
        "span.ArticleContributors__author-name",
        "[rel='author']",
    ]
    name_elems = []
    for sel in name_selectors:
        found = soup.select(sel)
        if found:
            name_elems = found
            break

    for el in name_elems:
        name = el.get_text(" ", strip=True)
        if name and name not in author_names:
            author_names.append(name)

    # Emails (mailto & elements with 'email' in class)
    for a in soup.select("a[href^='mailto:']"):
        mail = a["href"].replace("mailto:", "").strip()
        if mail and mail not in author_emails:
            author_emails.append(mail)

    email_like = soup.select("[class*='email'], [class*='Email']")
    for el in email_like:
        text = el.get_text(" ", strip=True)
        if "@" in text:
            for part in text.replace(";", " ").replace(",", " ").split():
                if "@" in part and part not in author_emails:
                    author_emails.append(part)

    # Addresses / affiliations
    aff_selectors = [
        "[class*='affiliation']",
        "li.affiliation",
        "div.author-info",
        "div.correspondence",
        "section#corresp",
    ]
    seen_aff_blocks = set()
    for sel in aff_selectors:
        for el in soup.select(sel):
            text = el.get_text(" ", strip=True)
            if not text:
                continue
            if text in seen_aff_blocks:
                continue
            seen_aff_blocks.add(text)
            author_addresses.append(text)

    return {
        "Abstract": abstract_text,
        "Keywords": keywords,
        "Author_name": "; ".join(author_names),
        "Author_email": "; ".join(author_emails),
        "Author_Address": " || ".join(author_addresses),
    }


def build_jit_article_data(
    in_csv: Path,
    out_csv: Path,
    sleep_seconds: float = 1.0,
) -> None:
    """Read Journal_of_IT.csv and write detailed article-level CSV."""
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    with in_csv.open(newline="", encoding="utf-8") as f_in, \
            out_csv.open("w", newline="", encoding="utf-8") as f_out:

        reader = csv.DictReader(f_in)
        fieldnames = [
            "URL",
            "Journal_Title",
            "Article_Title",
            "Volume_Issue",
            "Month_Year",
            "Abstract",
            "Keywords",
            "Author_name",
            "Author_email",
            "Author_Address",
        ]
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            url = (row.get("Issue_URL") or "").strip()
            if not url:
                continue

            issue_label = row.get("Title", "")  # e.g. 'Volume 39 Issue 4, December 2024'
            article_title = row.get("Issue_Title", "")

            # Split Volume_Issue vs Month_Year from issue_label
            volume_issue = issue_label
            month_year = ""
            if "," in issue_label:
                left, right = issue_label.split(",", 1)
                volume_issue = left.strip()
                month_year = right.strip()

            print(f"Scraping: {article_title} ({url})")
            try:
                details = parse_article_page(url)
            except Exception as e:
                print(f"  ERROR: {e}")
                details = {
                    "Abstract": "",
                    "Keywords": "",
                    "Author_name": "",
                    "Author_email": "",
                    "Author_Address": "",
                }

            out_row = {
                "URL": url,
                "Journal_Title": JOURNAL_NAME,
                "Article_Title": article_title,
                "Volume_Issue": volume_issue,
                "Month_Year": month_year,
            }
            out_row.update(details)
            writer.writerow(out_row)

            time.sleep(sleep_seconds)

    print(f"Finished. Wrote detailed data to {out_csv}")


# Convenience: set paths for your machine
base_dir = Path("/Users/keerthisagi/Documents/Journals/Journal_of_IT")
input_csv = base_dir / "Journal_of_IT.csv"
output_csv = base_dir / "JIT_article_data.csv"

print("STEP 2 ready. In a new cell, run:")
print("build_jit_article_data(input_csv, output_csv)")